# 04 — Topic Modelling with LDA

Inputs: `../data/grab_reviews_with_predictions.csv`  
Outputs: `../data/grab_reviews_topics.csv`

Steps:
1. Fit LDA on cleaned reviews (pure scikit-learn)
2. Inspect and label discovered topics (GrabFood, GrabCar, GrabPay, etc.)
3. Cross-tabulate sentiment x topic
4. Plot sentiment trend over time per topic

In [6]:
import warnings, os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/grab_reviews_with_predictions.csv', parse_dates=['date'])
print(f'Loaded {len(df):,} reviews')
df.head(2)

Loaded 5,333 reviews


,review_text,rating,date,author,source,review_clean,lang,sentiment,sentiment_id,vader_pred,bert_pred,bert_score
0,"Got scam by the app, put in an order waited ov...",1,2026-05-08 21:53:20,Ben Lim,google_play,"Got scam by the app, put in an order waited ov...",en,negative,0,negative,negative,0.885630
1,The app is well designed and easy to use . How...,5,2026-05-08 21:17:31,innocent boxer,google_play,The app is well designed and easy to use . How...,en,positive,2,positive,positive,0.856124


## 1. Fit LDA

In [7]:
N_TOPICS   = 10
N_TOP_WORDS = 12

docs = df['review_clean'].fillna('').tolist()

vectorizer = CountVectorizer(
    max_features=15000,
    stop_words='english',
    ngram_range=(1, 2),
    min_df=5,
)
dtm = vectorizer.fit_transform(docs)

lda = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=42,
    n_jobs=-1,
    max_iter=20,
)
lda.fit(dtm)

df['topic_id'] = lda.transform(dtm).argmax(axis=1)
print(f'Assigned {N_TOPICS} topics to {len(df):,} reviews')
df['topic_id'].value_counts().sort_index()

Assigned 10 topics to 5,333 reviews


topic_id
0    504
1    483
2    569
3    573
4    515
5    720
6    569
7    602
8    319
9    479
Name: count, dtype: int64

## 2. Inspect Topics

In [8]:
feature_names = vectorizer.get_feature_names_out()

for tid, topic_vec in enumerate(lda.components_):
    top_words = [feature_names[i] for i in topic_vec.argsort()[-N_TOP_WORDS:][::-1]]
    print(f'Topic {tid:2d}: {top_words}')

Topic  0: ['order', 'food', 'delivery', 'grab', 'cancelled', 'just', 'ordered', 'hour', 'rider', 'cancel', 'hours', 'waited']
Topic  1: ['ride', 'driver', 'grab', 'car', 'app', 'pay', 'minutes', 'payment', 'cash', 'price', 'time', 'charged']
Topic  2: ['service', 'customer', 'support', 'customer service', 'grab', 'help', 'issue', 'ai', 'app', 'account', 'bad', 'refund']
Topic  3: ['use', 'app', 'grab', 'just', 'easy', 'great', 'love', 'using', 'really', 'don', 'easy use', 'use app']
Topic  4: ['great', 'driver', 'nice', 'helpful', 'service', 'friendly', 'car', 'clean', 'safe', 'drivers', 'experience', 'great service']
Topic  5: ['food', 'app', 'delivery', 'convenient', 'easy', 'food delivery', 'grab', 'fast', 'reliable', 'use', 'rides', 'service']
Topic  6: ['good', 'app', 'delivery', 'drivers', 'service', 'time', 'food', 'price', 'grab', 'high', 'location', 'good service']
Topic  7: ['driver', 'cancel', 'time', 'drivers', 'grab', 'long', 'app', 'waiting', 'order', 'wait', 'booking', '

In [9]:
# Assign business-friendly labels after inspecting the words above
# Edit this mapping to match what you see in the topic words
TOPIC_LABELS = {
    0: 'GrabFood — Delivery',
    1: 'GrabCar — Driver Quality',
    2: 'App Performance / Bugs',
    3: 'GrabPay — Payment Issues',
    4: 'Pricing / Surge',
    5: 'Customer Service',
    6: 'GrabBike',
    7: 'Promotions / Rewards',
    8: 'Waiting Time / ETA',
    9: 'Safety',
}

df['topic_label'] = df['topic_id'].map(TOPIC_LABELS).fillna(
    df['topic_id'].apply(lambda x: f'Topic {x}')
)
print(df['topic_label'].value_counts())

topic_label
Customer Service            720
Promotions / Rewards        602
GrabPay — Payment Issues    573
App Performance / Bugs      569
GrabBike                    569
Pricing / Surge             515
GrabFood — Delivery         504
GrabCar — Driver Quality    483
Safety                      479
Waiting Time / ETA          319
Name: count, dtype: int64


## 3. Sentiment x Topic Cross-Tabulation

In [ ]:
pivot = (
    df.groupby(['topic_label', 'model_pred'])
    .size()
    .unstack(fill_value=0)
    .assign(total=lambda x: x.sum(axis=1))
    .sort_values('total', ascending=False)
)

for col in ['negative', 'neutral', 'positive']:
    if col in pivot.columns:
        pivot[f'{col}_pct'] = (pivot[col] / pivot['total'] * 100).round(1)

cols = [c for c in ['negative', 'neutral', 'positive', 'total'] if c in pivot.columns]
print(pivot[cols].head(15))

In [ ]:
top_topics = pivot.head(10).index.tolist()
plot_df = (
    df[df['topic_label'].isin(top_topics)]
    .groupby(['topic_label', 'model_pred'])
    .size()
    .reset_index(name='count')
)

palette = {'negative': '#e74c3c', 'neutral': '#f39c12', 'positive': '#2ecc71'}
pivot_plot = plot_df.pivot(index='topic_label', columns='model_pred', values='count').fillna(0)
pivot_plot = pivot_plot.reindex(top_topics)
cols_order = [c for c in ['negative', 'neutral', 'positive'] if c in pivot_plot.columns]

fig, ax = plt.subplots(figsize=(14, 6))
pivot_plot[cols_order].plot(
    kind='bar', stacked=True, ax=ax,
    color=[palette[c] for c in cols_order], edgecolor='white'
)
ax.set_title('Review Sentiment by Topic (Top 10)', fontsize=14)
ax.set_xlabel('Topic')
ax.set_ylabel('Review Count')
ax.legend(title='Sentiment')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../data/04_sentiment_by_topic.png', dpi=120)
plt.show()

## 4. Sentiment Trend Over Time per Topic

In [ ]:
sentiment_score = {'negative': -1, 'neutral': 0, 'positive': 1}
df['sent_score'] = df['model_pred'].map(sentiment_score)

top5 = df['topic_label'].value_counts().head(5).index.tolist()

fig, ax = plt.subplots(figsize=(14, 6))
for topic in top5:
    ts = (
        df[df['topic_label'] == topic]
        .set_index('date')
        .resample('ME')['sent_score']
        .mean()
        .rolling(3, min_periods=1)
        .mean()
    )
    ax.plot(ts.index, ts.values, marker='o', ms=3, lw=1.5, label=topic)

ax.axhline(0, color='black', lw=0.8, linestyle='--', alpha=0.4)
ax.set_title('Sentiment Trend Over Time by Topic (3-month rolling avg)', fontsize=13)
ax.set_xlabel('Month')
ax.set_ylabel('Avg Sentiment Score (-1 neg to +1 pos)')
ax.legend(loc='lower left', fontsize=9)
ax.set_ylim(-1, 1)
plt.tight_layout()
plt.savefig('../data/04_sentiment_trend_by_topic.png', dpi=120)
plt.show()

## 5. Save Final Dataset

In [ ]:
OUT = '../data/grab_reviews_topics.csv'
df.to_csv(OUT, index=False)
print(f'Saved {len(df):,} rows to {OUT}')
df[['review_clean', 'sentiment', 'model_pred', 'topic_label', 'date']].head()